In [8]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

# Wczytujemy dane treningowe i testowe
train = fetch_20newsgroups(subset='train', shuffle=True, random_state=42)
test  = fetch_20newsgroups(subset='test',  shuffle=True, random_state=42)

print(f"Liczba dokumentów treningowych: {len(train.data)}")
print(f"Liczba dokumentów testowych:    {len(test.data)}")
print(f"Liczba klas:                    {len(train.target_names)}")
print(f"\nKlasy:\n{train.target_names}")

Liczba dokumentów treningowych: 11314
Liczba dokumentów testowych:    7532
Liczba klas:                    20

Klasy:
['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


In [9]:
from collections import Counter

# Klasa większościowa — najprostsza możliwa "prognoza"
most_common_class = Counter(train.target).most_common(1)[0][0]
baseline_preds = np.full(len(test.target), fill_value=most_common_class)
baseline_acc = np.mean(baseline_preds == test.target)

print(f"Dokładność modelu bazowego (klasa większościowa): {baseline_acc:.4f}")
print(f"Czyli ~{baseline_acc*100:.1f}% — tyle daje 'zgadywanie' najczęstszej klasy")

Dokładność modelu bazowego (klasa większościowa): 0.0530
Czyli ~5.3% — tyle daje 'zgadywanie' najczęstszej klasy


In [7]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.pipeline import Pipeline
from nltk.stem.snowball import SnowballStemmer
import nltk

# Pobierz zasoby NLTK (tylko przy pierwszym uruchomieniu)
nltk.download('stopwords')

stemmer = SnowballStemmer("english", ignore_stopwords=True)

class StemmedCountVectorizer(CountVectorizer):
    """CountVectorizer z wbudowanym stemmingiem."""
    def build_analyzer(self):
        analyzer = super().build_analyzer()
        return lambda doc: [stemmer.stem(w) for w in analyzer(doc)]

def build_pipeline(classifier, use_stemming=False, stop_words='english'):
    """
    Buduje pipeline: wektoryzacja → TF-IDF → klasyfikator.
    
    Parametry:
      classifier   – obiekt klasyfikatora sklearn
      use_stemming – czy używać stemmingu (wolniejsze, często lepsze)
      stop_words   – 'english' lub None
    """
    if use_stemming:
        vect = StemmedCountVectorizer(stop_words=stop_words, ngram_range=(1, 2))
    else:
        vect = CountVectorizer(stop_words=stop_words, ngram_range=(1, 2))
    
    return Pipeline([
        ('vect', vect),
        ('tfidf', TfidfTransformer(use_idf=True)),
        ('clf', classifier)
    ])

Fitting 5 folds for each of 11 candidates, totalling 55 fits


TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.

The exit codes of the workers are {SIGKILL(-9)}
Detailed tracebacks of the workers should have been printed to stderr in the executor process if faulthandler was not disabled.

In [ ]:
best_model = grid.best_estimator_
y_pred = best_model.predict(test.data)

print("\n--- Raport końcowy ---")
print(f"Dokładność testowa: {accuracy_score(test.target, y_pred):.4f}")
print("\nSzczegółowy raport:\n")
print(classification_report(test.target, y_pred, target_names=test.target_names))

In [6]:
## Wnioski i analiza
1. **Model bazowy:** Wynik ok. 0.05 (5%) potwierdza, że klasy są w miarę równomiernie rozłożone (1/20 = 0.05).
2. **Wpływ etapu wektoryzacji:** Zastosowanie `ngram_range=(1,2)` w `TfidfVectorizer` pozwoliło modelowi lepiej rozumieć kontekst wyrażeń.
3. **Porównanie modeli:** - `MultinomialNB` okazał się zaskakująco skuteczny i szybki.
   - `LogisticRegression` osiągnął prawdopodobnie najwyższą dokładność dzięki możliwości regularyzacji (parametr C).
   - `RandomForest` był znacznie wolniejszy i mniej efektywny w tej przestrzeni cech.
4. **Co najbardziej poprawiło wynik:** Największy skok jakościowy dała zmiana `CountVectorizer` na `TfidfVectorizer` oraz dodanie bigramów, co skuteczniej odseparowało tematykę wiadomości.

SyntaxError: invalid syntax (1465670318.py, line 2)